# **DFC-3.3-22M**
**Перед проверкой не забудьте прочитать README.MD!**

**Общая информация:**
1) Архитектура: ResNet34 + SE(Squeeze-and-Excitation) + SRM-фильтры
2) Обучение: при помощи сбалансированная кроссвалидации на 5 фолдах, 20 эпох на каждой, применено сглаживание весов (EMA), AMP для увеличения скорости обучения
3) Для борьбы с дисбалансом изменены веса штрафов
4) На валидации и тестировании используется TTA

## **1. Импорты**
Тут мало чего комментировать, просто импорты разных модулей

In [ ]:
import copy
import gc
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm 

import torch
import torch.nn as nn
from torchinfo import summary
import torch.nn.functional as F
from torch.nn.modules.loss import BCEWithLogitsLoss
from torchvision.transforms.v2._container import Compose
from torch.utils.data import DataLoader, Dataset
# from torchvision.transforms.v2 import (
#     Compose, ToImage, ToDtype, Normalize,
# )
from torchvision.transforms import v2
from torchvision.transforms import InterpolationMode

from torch.optim.swa_utils import AveragedModel

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## **2. Конфиги**
Это словарь с константами, используемыми для обучения. 

**Обозначения:**
- "dataset_path" — путь к корневой директории датасета
- "train_images_dirname" — папка с обучающими изображениями
- "test_images_dirname" — папка с тестовыми изображениями
- "train_csv_name" — CSV-файл с разметкой тренировочных данных
- "image_size" — размер изображения (высота и ширина)
- "batch_size" — размер батча
- "num_workers" — число процессов для загрузки данных (ускоряет DataLoader)
- "epochs" — количество эпох обучения
- "lr" — learning rate (скорость обучения)
- "weight_decay" — коэффициент L2-регуляризации (борьба с переобучением)
- "seed" — "зёрношко" для псевдослучайности
- "n_splits" — количество фолдов
- "use_amp" — использование AMP (mixed precision) для ускорения и экономии памяти
- "max_grad_norm" — ограничение нормы градиента (gradient clipping, стабилизирует обучение)
- "save_dir" — директория для чекпоинтов модели
- "oof_path" — файл для OOF-предсказаний
- "test_probs_path" — вероятности для тестовой выборки
- "submission_path" — путь к сабмиту


In [ ]:
CONFIG = { 
    "dataset_path": "/kaggle/input/datasets/daniilkrasnovvv/yl-dataset3/dataset",
    "train_images_dirname": "train_images",
    "test_images_dirname": "test_images",
    "train_csv_name": "train_solution.csv",

    "image_size": 256,
    "batch_size": 32,
    "num_workers": 2,

    "epochs": 20,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "seed": 42,

    "n_splits": 5,
    "use_amp": True,
    "max_grad_norm": 1.0,

    "save_dir": "/kaggle/working/dfc3_21m_checkpoints",
    "oof_path": "/kaggle/working/oof_predictions_dfc3_21m.csv",
    "test_probs_path": "/kaggle/working/test_probabilities_dfc3_21m.csv",
    "submission_path": "/kaggle/working/submission_dfc3_21m.csv",
}

In [ ]:
# YL-Dataset mean и std по каналам (R, G, B)
MEAN = [0.519, 0.428, 0.384]
STD = [0.286, 0.264, 0.264]

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
dataset_path = CONFIG["dataset_path"]
train_images_path = os.path.join(dataset_path, CONFIG["train_images_dirname"])
test_images_path = os.path.join(dataset_path, CONFIG["test_images_dirname"])
train_csv_path = os.path.join(dataset_path, CONFIG["train_csv_name"])

print("dataset_path exists:", os.path.exists(dataset_path))
print("train_images_path exists:", os.path.exists(train_images_path))
print("test_images_path exists:", os.path.exists(test_images_path))
print("train_csv exists:", os.path.exists(train_csv_path))

## **3. Трансформы**
Чтобы искусственно расширить датасет, я применяю 

- к трейновым картинкам (get_train_transform):
1) Случайная обрезка
2) Горизонтальный разворот 
3) Симуляция размытия (сжатие и разжатие)
4) Нормализация

- к тестовым и валидационным (get_eval_transform):
3) Нормализация

In [ ]:
def get_train_transform():
    return v2.Compose([
        v2.ToImage(),

        v2.RandomResizedCrop(
            size=(CONFIG["image_size"], CONFIG["image_size"]),
            scale=(0.97, 1.00),
            ratio=(0.98, 1.02),
            interpolation=InterpolationMode.BILINEAR,
            antialias=True,
        ),

        v2.RandomHorizontalFlip(p=0.5),

        v2.RandomApply([
            v2.JPEG(quality=(60, 95))
        ], p=0.20),

        v2.RandomApply([
            v2.Resize(
                size=(int(CONFIG["image_size"] * 0.8), int(CONFIG["image_size"] * 0.8)),
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            ),
            v2.Resize(
                size=(CONFIG["image_size"], CONFIG["image_size"]),
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            ),
        ], p=0.15),

        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=MEAN, std=STD),
    ])

def get_eval_transform():
    return Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=MEAN, std=STD),
    ])

## **4. Обработка данных**

### **Класс датасета**
Это дочерний класс от Dataset для работы с картинками

Параметры конструктора:
1) df - DataFrame датасета, сгенерированный с помощью build_train_dataframe или build_train_dataframe
2) train_images_folder - путь до папки с тренировочными картинками
3) test_images_folder - путь до папки с тестовыми картинками
4) transform - Compose объект
5) is_test - флаг того, работаем мы с тестовыми картинками (True) или тренировочными (False)

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        train_images_folder: str,
        test_images_folder: str,
        transform=None,
        is_test: bool = False,
    ):
        self.df = df.reset_index(drop=True).copy()
        self.train_images_folder = train_images_folder
        self.test_images_folder = test_images_folder
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_name = row["ImageName"]

        if self.is_test:
            image_path = os.path.join(self.test_images_folder, image_name)
        else:
            source = row.get("source", "train")
            if source == "pseudo":
                image_path = os.path.join(self.test_images_folder, image_name)
            else:
                image_path = os.path.join(self.train_images_folder, image_name)

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        if self.is_test:
            return image, int(row["id"])

        return image, torch.tensor(float(row["IsDeepfake"]), dtype=torch.float32)

### **Сборка данных**

In [ ]:
def load_train_with_pseudo(dataset_path: str, csv_name: str):
    csv_path = os.path.join(dataset_path, csv_name)
    df = pd.read_csv(csv_path)

    required_cols = {"id", "ImageName", "IsDeepfake", "source"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"В CSV не хватает колонок: {missing}")

    df = df.copy()
    df["id"] = df["id"].astype(int)
    df["ImageName"] = df["ImageName"].astype(str)
    df["IsDeepfake"] = df["IsDeepfake"].astype(int)
    df["source"] = df["source"].astype(str).str.lower()

    train_df = df[df["source"] == "train"].reset_index(drop=True)
    pseudo_df = df[df["source"] == "pseudo"].reset_index(drop=True)

    return train_df, pseudo_df

In [ ]:
def build_test_dataframe(dataset_path: str,
                         test_images_dirname: str = "test_images") -> pd.DataFrame:
    test_images_path = os.path.join(dataset_path, test_images_dirname)

    rows = []
    skipped = []

    for fname in os.listdir(test_images_path):
        fpath = os.path.join(test_images_path, fname)
        if not os.path.isfile(fpath):
            continue

        ext = Path(fname).suffix.lower()
        if ext != '.jpg':
            continue

        stem = Path(fname).stem
        try:
            file_id = int(stem)
            rows.append((file_id, fname))
        except ValueError:
            skipped.append(fname)

    if skipped:
        raise ValueError(f"Пропущены файлы с нечисловым stem. Первые 10: {skipped[:10]}")

    test_df = pd.DataFrame(rows, columns=["id", "ImageName"])
    test_df = test_df.sort_values("id").reset_index(drop=True)
    return test_df

In [ ]:
full_train_df, _ = load_train_with_pseudo(
    dataset_path=CONFIG["dataset_path"],
    csv_name="train_solution_mod_with_confident_data.csv",
)

test_df = build_test_dataframe(
    dataset_path=CONFIG["dataset_path"],
    test_images_dirname=CONFIG["test_images_dirname"],
)

In [ ]:
print(full_train_df.head())
print()
print(test_df.head())
print()
print("train shape:", full_train_df.shape)
print("test shape:", test_df.shape)
print("positive ratio:", full_train_df["IsDeepfake"].mean())

### **Полезные функции**
А именно:
1) gen_datasets - создание датасета по датафрейму фолда
2) gen_dataloaders - создание даталоудеров

In [ ]:
def gen_datasets(fold_train_df, fold_val_df):
    train_dataset = DeepfakeDataset(
        df=fold_train_df,
        train_images_folder=train_images_path,
        test_images_folder=test_images_path,
        transform=get_train_transform(),
        is_test=False,
    )
    val_dataset = DeepfakeDataset(
        df=fold_val_df,
        train_images_folder=train_images_path,
        test_images_folder=test_images_path,
        transform=get_eval_transform(),
        is_test=False,
    )
    test_dataset = DeepfakeDataset(
        df=test_df,
        train_images_folder=train_images_path,
        test_images_folder=test_images_path,
        transform=get_eval_transform(),
        is_test=True,
    )

    return train_dataset, val_dataset, test_dataset

In [ ]:
def gen_dataloaders(train_dataset, val_dataset, test_dataset):
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader

## **5. Архитектура SE-ResNet34 + SRM-фильтры**
Как уже было сказано ранее, в основе DFC-3.2 лежит архитектура ResNet34 с Squeeze-and-Excitation блоками. 

- Я добавил SE, потому что:
1) Помогает сети сконцентрироваться на более информативных каналах изображения в плане идентификации дипфейков
2) В датасете присутствует шум
3) Почти бесплатная по GPU оптимизация, которая рекомендуется для моей архитектуры

- Я добавил SRM-фильтры, для поиска артефактов генерации данных. Веса подобраны учёными по ImageNet и другим датасетам, но ментор сказал, что это не нарушение правил, так как по сути это просто какая-то необучаемая константа

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class SEResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, reduction: int = 16):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.se = SEBlock(out_channels, reduction)

        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)

        out += identity
        out = self.relu(out)
        return out


class MultiSRMLayer(nn.Module):
    def __init__(self, clamp_value=3.0):
        super().__init__()

        kernels = torch.tensor([
            [[0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0],
             [0, 1, -2, 1, 0],
             [0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0]],

            [[0, 0, 0, 0, 0],
             [0, -1, 2, -1, 0],
             [0, 2, -4, 2, 0],
             [0, -1, 2, -1, 0],
             [0, 0, 0, 0, 0]],

            [[-1, 2, -2, 2, -1],
             [2, -6, 8, -6, 2],
             [-2, 8, -12, 8, -2],
             [2, -6, 8, -6, 2],
             [-1, 2, -2, 2, -1]]
        ], dtype=torch.float32)

        kernels[0] /= 2.0
        kernels[1] /= 4.0
        kernels[2] /= 12.0

        weight = kernels.unsqueeze(1).repeat(3, 1, 1, 1)   # 9, 1, 5, 5
        self.register_buffer("weight", weight)

        self.clamp = nn.Hardtanh(min_val=-clamp_value, max_val=clamp_value)

    def forward(self, x):
        x = F.conv2d(x, self.weight, bias=None, stride=1, padding=2, groups=3)
        x = self.clamp(x)
        return x


def make_stage(in_channels: int, out_channels: int, block_count: int, stride: int):
    layers = [SEResidualBlock(in_channels, out_channels, stride=stride)]
    for _ in range(block_count - 1):
        layers.append(SEResidualBlock(out_channels, out_channels, stride=1))
    return nn.Sequential(*layers)


class DFC(nn.Module):
    def __init__(self, num_classes: int = 1, dropout_p: float = 0.2):
        super().__init__()

        self.rgb_stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.rgb_layer1 = make_stage(64, 64, block_count=3, stride=1)
        self.rgb_layer2 = make_stage(64, 128, block_count=4, stride=2)
        self.rgb_layer3 = make_stage(128, 256, block_count=6, stride=2)
        self.rgb_layer4 = make_stage(256, 512, block_count=3, stride=2)

        self.srm = MultiSRMLayer()

        self.hf_stem = nn.Sequential(
            nn.Conv2d(9, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.hf_layer1 = make_stage(32, 32, block_count=1, stride=1)
        self.hf_layer2 = make_stage(32, 64, block_count=2, stride=2)
        self.hf_layer3 = make_stage(64, 128, block_count=2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.head = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(512 + 128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(128, num_classes)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
            elif isinstance(m, nn.BatchNorm2d):
                if m.weight is not None:
                    nn.init.constant_(m.weight, 1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rgb = self.rgb_stem(x)
        rgb = self.rgb_layer1(rgb)
        rgb = self.rgb_layer2(rgb)
        rgb = self.rgb_layer3(rgb)
        rgb = self.rgb_layer4(rgb)
        rgb = self.avgpool(rgb)
        rgb = torch.flatten(rgb, 1)

        hf = self.srm(x)
        hf = self.hf_stem(hf)
        hf = self.hf_layer1(hf)
        hf = self.hf_layer2(hf)
        hf = self.hf_layer3(hf)
        hf = self.avgpool(hf)
        hf = torch.flatten(hf, 1)

        feats = torch.cat([rgb, hf], dim=1)
        out = self.head(feats)
        return out

In [ ]:
model = DFC()
print(summary(model))

## **6. Вспомогательные функции**

А именно:
1) *build_criterion* - создание лосса

    Я решил использовать BCEWithLogitsLoss, так как хотел, чтобы модель выдавала уверенные предсказания.
    Возникла проблема: из-за дизбаланса в датасете модель делала константные предсказания. Я попытался это решить добавлением повышенного веса ошибки на редком классе (1), но это лишь поменяло ситуацию со всех 0 на все 1. Решением было брать не стандартную математическую формулу веса ошибки по соотношению семплов 0 и 1, а квадратный корень от данного числа
2) *calculate_metrics_from_probs* - подсчёт метрик качества по вероятностям модели
3) *find_best_threshold* - подбор порога уверенности модели. Максимизируется F1-score
4) *create_submission* - создание DataFrame-а для сабмита

In [ ]:
def build_criterion(train_df: pd.DataFrame, device) -> tuple[BCEWithLogitsLoss, float]:
    from math import sqrt

    num_pos = int(train_df["IsDeepfake"].sum())
    num_neg = int(len(train_df) - num_pos)

    pos_weight_value = sqrt(num_neg / max(num_pos, 1))

    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    return criterion, pos_weight_value

In [ ]:
def find_best_threshold(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.01)

    best_thr = 0.5
    best_f1 = -1.0

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_thr = thr

    return best_thr, best_f1

In [ ]:
def calculate_metrics_from_probs(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

    try:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
    except ValueError:
        metrics["roc_auc"] = np.nan

    return metrics

In [ ]:
def create_submission(ids, predictions) -> pd.DataFrame:
    return pd.DataFrame({
        "id": np.asarray(ids, dtype=int),
        "target_feature": np.asarray(predictions, dtype=int),
    })

### **Валидация, тестирование**
1) *collect_predictions* - сбор предсказаний модели на валидации, выдаёт вероятности
2) *evaluate_loss* - подсчёт лосса
3) *predict_test* - создание датафрейма с вероятностями для тестовых данных

In [ ]:
@torch.inference_mode()
def collect_predictions(model, loader, device, use_tta=True):
    model.eval()
    all_probs = []
    all_targets = []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        logits = model(images).squeeze(1)

        if use_tta:
            images_flip = torch.flip(images, dims=[3])  # horizontal flip
            logits_flip = model(images_flip).squeeze(1)

            logits = (logits + logits_flip) / 2.0 # Среднее по логитам
        
        probs = torch.sigmoid(logits)
        probs = probs.detach().cpu().numpy().reshape(-1)
        
        all_probs.extend(probs)
        all_targets.extend(targets.cpu().numpy())

    return np.array(all_probs, dtype=float), np.array(all_targets, dtype=float)

In [ ]:
@torch.inference_mode()
def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True).unsqueeze(1)

        logits = model(images)
        loss = criterion(logits, targets)

        total_loss += loss.item() * images.size(0)

    return total_loss / max(len(loader.dataset), 1)

In [ ]:
@torch.inference_mode()
def predict_test(model, loader, device, use_tta=True):
    model.eval()
    model.to(device)

    all_probs = []
    all_ids = []

    for images, ids in tqdm(loader, desc="Predict test"):
        images = images.to(device, non_blocking=True)

        logits = model(images).squeeze(1)

        if use_tta:
            images_flip = torch.flip(images, dims=[3])  # horizontal flip
            logits_flip = model(images_flip).squeeze(1)

            logits = (logits + logits_flip) / 2.0 # Среднее по логитам
        
        probs = torch.sigmoid(logits)
        probs = probs.detach().cpu().numpy().reshape(-1)

        all_probs.extend(probs.tolist())
        all_ids.extend([int(x) for x in ids])

    pred_df = pd.DataFrame({
        "id": all_ids,
        "probability": all_probs
    }).sort_values("id").reset_index(drop=True)

    return pred_df

### **Методы для обучения**
1) *train_one_epoch* - проход по одной эпохе

Параметры:
- model — обучаемая модель
- ema_model — EMA модель
- loader — train лоадер
- scaler — GradScaler (AMP)
- use_amp — флаг использования AMP
- max_grad_norm — clipping градиентов

2) *build_ema_model* - сборка EMA-модели из обычной. decay=0.9995 - стандарт для CV
3) *build_model* - сборка всех необходимых компонент для создания и обучения модели: model, ema_model, criterion, optimizer, scheduler, scaler

In [ ]:
def train_one_epoch(model, ema_model, loader, criterion, optimizer, 
                    scheduler, device, scaler, use_amp=True, max_grad_norm=None):
    model.train()
    running_loss = 0.0

    progress = tqdm(loader, desc="train", leave=False)

    for images, targets in progress:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(use_amp and device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        if max_grad_norm is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        scaler.step(optimizer)
        scaler.update()

        ema_model.update_parameters(model)

        running_loss += loss.item() * images.size(0)
        progress.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    epoch_loss = running_loss / max(len(loader.dataset), 1)
    return epoch_loss

In [ ]:
def build_ema_model(model, device, decay=0.9995):
    def ema_avg_fn(averaged_model_parameter, model_parameter, num_averaged):
        return decay * averaged_model_parameter + (1.0 - decay) * model_parameter

    ema_model = AveragedModel(model, avg_fn=ema_avg_fn).to(device)
    return ema_model

In [ ]:
def build_model(fold_train_df):
    model = DFC().to(device)

    ema_model = build_ema_model(model, device)

    criterion, _ = build_criterion(
        fold_train_df,
        device=device,
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CONFIG["epochs"],
        eta_min=CONFIG["lr"] * 0.05,
    )
    scaler = torch.amp.GradScaler(enabled=(CONFIG["use_amp"] and device.type == "cuda"))

    return model, ema_model, criterion, optimizer, scheduler, scaler

In [ ]:
def fit_fold(model, ema_model, train_loader, val_loader, criterion, 
             optimizer, scheduler, device, scaler, 
             history, fold_idx):
    best_f1 = -1.0
    best_threshold = 0.5
    best_state_dict = None

    for epoch in range(1, CONFIG["epochs"] + 1):
        print(f"\nFold {fold_idx} | Epoch {epoch}/{CONFIG['epochs']}")

        train_loss = train_one_epoch(
            model=model,
            ema_model=ema_model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            scaler=scaler,
            use_amp=CONFIG["use_amp"],
            max_grad_norm=CONFIG["max_grad_norm"],
        )

        val_loss = evaluate_loss(
            model=ema_model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        if scheduler is not None:
            scheduler.step()

        val_probs, val_targets = collect_predictions(
            model=ema_model,
            loader=val_loader,
            device=device,
        )

        val_threshold, val_f1 = find_best_threshold(val_targets, val_probs)
        val_metrics = calculate_metrics_from_probs(
            y_true=val_targets,
            y_prob=val_probs,
            threshold=val_threshold,
        )

        print(
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"precision={val_metrics['precision']:.4f} | "
            f"recall={val_metrics['recall']:.4f} | "
            f"roc_auc={val_metrics['roc_auc']:.4f} | "
            f"thr={val_threshold:.2f}"
        )

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_metrics["f1"])
        history["val_precision"].append(val_metrics["precision"])
        history["val_recall"].append(val_metrics["recall"])
        history["val_roc_auc"].append(val_metrics["roc_auc"])
        history["best_threshold"].append(val_threshold)

        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_threshold = val_threshold
            best_state_dict = copy.deepcopy(ema_model.state_dict())

            ckpt_path = os.path.join(CONFIG["save_dir"], f"dfc3_21m_fold_{fold_idx}.pt")
            torch.save({
                "model_state_dict": best_state_dict,
                "best_f1": best_f1,
                "best_threshold": best_threshold,
                "config": CONFIG,
            }, ckpt_path)
            print(f"Saved best checkpoint: {ckpt_path}")
    
    return best_state_dict, best_f1, best_threshold

## **7. Кросс-валидационная часть**

Создание сплиттера, анализ того, как разбиваются данные

In [ ]:
skf = StratifiedKFold(
    n_splits=CONFIG["n_splits"],
    shuffle=True,
    random_state=CONFIG["seed"],
)

folds = list(skf.split(full_train_df, full_train_df["IsDeepfake"]))

In [ ]:
# Текущее состояние
print("Number of folds:", len(folds))

for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):
    fold_train_mean = full_train_df.iloc[train_idx]["IsDeepfake"].mean()
    fold_val_mean = full_train_df.iloc[val_idx]["IsDeepfake"].mean()
    print(
        f"Fold {fold_idx}: "
        f"train_size={len(train_idx)}, val_size={len(val_idx)}, "
        f"train_pos_ratio={fold_train_mean:.4f}, val_pos_ratio={fold_val_mean:.4f}"
    )

## **8. Обучение**
**Pipeline обучения.**
Для каждого фолда (5 штук) цикл из 20 эпох. По итогу выбирается лучшая модель, которая далее предсказывается часть train_images и test_images. Логируются метрики на валидации. Лучшая модель сохраняется в Config['save_dir']. 


In [ ]:
os.makedirs(CONFIG["save_dir"], exist_ok=True)

oof_df = full_train_df[["id", "ImageName", "IsDeepfake"]].copy()
oof_df["fold"] = -1
oof_df["oof_probability"] = np.nan

test_prob_df = test_df[["id", "ImageName"]].copy()
test_prob_columns = []

fold_summaries = []
histories = {}

In [ ]:
for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):
    print("=" * 100)
    print(f"FOLD {fold_idx}/{CONFIG['n_splits']}")

    base_fold_train_df = full_train_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = full_train_df.iloc[val_idx].reset_index(drop=True)
    
    # Псевдолейблы добавляем только в train
    fold_train_df = base_fold_train_df
    
    fold_train_df = fold_train_df.sample(
        frac=1.0,
        random_state=CONFIG["seed"] + fold_idx,
    ).reset_index(drop=True)

    train_dataset, val_dataset, test_dataset = gen_datasets(fold_train_df, fold_val_df)
    train_loader, val_loader, test_loader = gen_dataloaders(train_dataset, val_dataset, test_dataset)

    model, ema_model, criterion, optimizer, scheduler, scaler = build_model(base_fold_train_df)

    print(f"Fold {fold_idx} train size: {len(fold_train_df)} | val size: {len(fold_val_df)}")

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_f1": [],
        "val_precision": [],
        "val_recall": [],
        "val_roc_auc": [],
        "best_threshold": [],
    }

    best_state_dict, best_f1, best_threshold = fit_fold(model, ema_model, train_loader, 
                                                        val_loader, criterion, 
                                                        optimizer, scheduler, 
                                                        device, scaler, 
                                                        history, fold_idx)
    
    histories[f"fold_{fold_idx}"] = pd.DataFrame(history)

    ema_model.load_state_dict(best_state_dict)

    val_probs, val_targets = collect_predictions(
        model=ema_model,
        loader=val_loader,
        device=device,
    )
    final_metrics = calculate_metrics_from_probs(
        y_true=val_targets,
        y_prob=val_probs,
        threshold=best_threshold,
    )

    oof_df.loc[val_idx, "fold"] = fold_idx
    oof_df.loc[val_idx, "oof_probability"] = val_probs

    fold_test_prob_df = predict_test(
        model=ema_model,
        loader=test_loader,
        device=device,
        use_tta=False,
    )
    column_name = f"fold_{fold_idx}_probability"
    test_prob_df = test_prob_df.merge(
        fold_test_prob_df.rename(columns={"probability": column_name}),
        on="id",
        how="left",
    )
    test_prob_columns.append(column_name)

    fold_summaries.append({
        "fold": fold_idx,
        "best_f1": best_f1,
        "best_threshold": best_threshold,
        "precision": final_metrics["precision"],
        "recall": final_metrics["recall"],
        "roc_auc": final_metrics["roc_auc"],
    })

    del model, ema_model, optimizer, scheduler, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## **9. Финальный submit**

Наконец то, ради чего всё затевалось: предсказания тестовых данных! 

Алгоритм:
1) Подбирается лучший порог на train предсказаниях, которые были сделаны во время обучения
2) С помощью предсказанных вероятностей на test и порога на train, делаются предсказание
3) Сохранение submition.csv

In [ ]:
fold_summary_df = pd.DataFrame(fold_summaries)
display(fold_summary_df)

oof_valid = oof_df.dropna(subset=["oof_probability"]).copy()
assert len(oof_valid) == len(full_train_df), "Не для всех объектов получены OOF-предсказания"

global_best_threshold, global_best_f1 = find_best_threshold(
    oof_valid["IsDeepfake"].values,
    oof_valid["oof_probability"].values,
)
oof_metrics = calculate_metrics_from_probs(
    oof_valid["IsDeepfake"].values,
    oof_valid["oof_probability"].values,
    threshold=global_best_threshold,
)

print("OOF best threshold:", global_best_threshold)
print("OOF F1:", global_best_f1)
print("OOF metrics:", oof_metrics)

oof_df.to_csv(CONFIG["oof_path"], index=False)
print(f"Saved {CONFIG['oof_path']}")

test_prob_df["probability"] = test_prob_df[test_prob_columns].mean(axis=1)
test_prob_df = test_prob_df.sort_values("id").reset_index(drop=True)

test_prob_df["target_feature"] = (
    test_prob_df["probability"] >= global_best_threshold
).astype(int)

submission = create_submission(
    ids=test_prob_df["id"].values,
    predictions=test_prob_df["target_feature"].values,
)

test_prob_df.to_csv(CONFIG["test_probs_path"], index=False)
submission.to_csv(CONFIG["submission_path"], index=False)

display(test_prob_df.head())
display(submission.head())

print(f"Saved {CONFIG['test_probs_path']}")
print(f"Saved {CONFIG['submission_path']}")